# Infinaxis xyz geometry
This notebooks goes through how to use fullcontrol's existing geometry functions with infinaxis points.

In [ ]:
if 'google.colab' in str(get_ipython()):
  !pip install git+https://github.com/FullControlXYZ/fullcontrol --quiet
import lab.fullcontrol.infinaxis as fci

EW = 0.6
EH = 0.3

head_chain = []
bed_chain = [fci.Axis(name='C')]
Point = fci.configure_point(head_chain, bed_chain)

gcode_controls = fci.GcodeControls(
    head_chain=head_chain,
    bed_chain=bed_chain,
    initialization_data={'extrusion_width': EW, 'extrusion_height': EH},
)


The fullcontrol geometry functions are available through `fci.xyz_geom` or alternatively can just be called through a separate instance of normal fullcontrol `fc.`. For instance `fci.xyz_geom.circleXY` is equivalent to `fc.circleXY`.

Once the xyz steps have been created they can be converted to the configured infinaxis points `Point` through the function `fci.xyz_add_axes` function.

In [ ]:
xyz_steps = fci.xyz_geom.circleXY(
    fci.xyz_geom.Point(x=0, y=0, z=EH / 2),
    radius=20,
    start_angle=0,
    segments=72,
)

steps = fci.xyz_add_axes(xyz_steps, Point)
steps.insert(0, fci.Printer(print_speed=1200))

for i, step in enumerate(steps):
    if isinstance(step, fci.Point):
        step.c = i * 5
        step.color = [(step.c % 360) / 360, 0, 1 - ((step.c % 360) / 360)]


In [ ]:
fig = fci.transform(
    steps,
    'fig',
    fci.PlotControls(color_type='manual', style='line', zoom=0.75),
    show_tips=False,
)
fig.show()

gcode = fci.transform(steps, 'gcode', gcode_controls)
print('first ten gcode lines:\n' + '\n'.join(gcode.split('\n')[:10]))
